The `SparkSession` is your single entry point into everything Spark. Understanding what it is, how it relates to the older `SparkContext`, and how Spark executes your code — lazily building a DAG and only running on an action — is the foundation for writing correct and efficient Spark programs.

## SparkSession — The Entry Point

`SparkSession` was introduced in Spark 2.0 to unify three previously separate entry points:

| Old API (Spark 1.x) | Replaced by |
|---|---|
| `SparkContext` | Still exists — `spark.sparkContext` |
| `SQLContext` | Merged into `SparkSession` |
| `HiveContext` | Merged into `SparkSession` |

With `SparkSession` you get one object that handles everything: creating DataFrames, running SQL, reading data, and accessing the underlying `SparkContext` when needed.

A `SparkSession` per application is the norm. On **Databricks**, one is always pre-created for you as `spark`. In a standalone script or notebook you build one with the builder pattern.

## SparkContext vs SparkSession

`SparkContext` (`sc`) is the lower-level object — it represents the connection to the cluster and is still the layer that actually manages executors, tasks, and RDDs.

`SparkSession` (`spark`) wraps `SparkContext` and adds the DataFrame/SQL layer on top.

```
SparkSession
  └── SparkContext          ← cluster connection, RDD API
       ├── DAG Scheduler
       ├── Task Scheduler
       └── Executor backends
```

In practice:
- Use `spark` (SparkSession) for everything DataFrame/SQL
- Access `spark.sparkContext` only when you need the low-level RDD API or to read Spark configuration

## Creating a SparkSession

The builder pattern gives you full control over the session configuration:

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MyApp")              # shows in Spark UI and logs
    .master("local[*]")            # local mode: use all CPU cores
    .config("spark.sql.shuffle.partitions", "8")   # tune for small data
    .config("spark.driver.memory", "2g")
    .getOrCreate()                 # reuse existing session if one exists
)

print(spark)
print(f"Spark version : {spark.version}")
print(f"App name      : {spark.sparkContext.appName}")
print(f"Master        : {spark.sparkContext.master}")

### Key SparkSession Entry Points

| Entry point | What you get |
|---|---|
| `spark.read` | `DataFrameReader` — load data from files, JDBC, Delta, etc. |
| `spark.sql("...")` | Run a SQL string, returns a DataFrame |
| `spark.range(n)` | DataFrame of integers 0 to n-1 — useful for testing |
| `spark.createDataFrame(data)` | Build a DataFrame from a Python list or pandas DataFrame |
| `spark.catalog` | Inspect and manage tables, databases, functions |
| `spark.conf` | Read and set runtime configuration |
| `spark.sparkContext` | Access the underlying `SparkContext` (RDD API, accumulators) |

## Lazy Evaluation

Spark's most important design principle: **transformations are lazy**.

When you call `.filter()`, `.select()`, `.groupBy()`, or any other transformation, Spark does **not** execute anything. It records the operation as a step in a logical plan. Execution is deferred until you call an **action**.

![Lazy Evaluation](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-lazy-evaluation.svg)

**Why lazy evaluation matters:**
- Spark can see the entire pipeline before running it — the **Catalyst optimizer** can reorder, merge, and eliminate steps
- A filter pushed before a join reads far less data
- Multiple transformations on the same dataset can be fused into a single pass (pipelining)
- You avoid unnecessary computation — if you define a transformation but never call an action, nothing runs

## Transformations vs Actions

| | Transformations | Actions |
|---|---|---|
| **Execution** | Lazy — just update the plan | Eager — trigger DAG execution |
| **Return type** | Another DataFrame / RDD | A result (value, file, printed output) |
| **Examples** | `.filter()` `.select()` `.join()` `.groupBy()` `.withColumn()` | `.show()` `.count()` `.collect()` `.take(n)` `.write` |

### Narrow vs Wide Transformations

Transformations are further classified by how data flows between partitions:

![Narrow vs Wide Transformations](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-narrow-wide-transforms.svg)

**Narrow** — each output partition depends on **one** input partition. No data crosses the network. All narrow transformations in a sequence are pipelined into a single stage.
- Examples: `map`, `filter`, `select`, `withColumn`, `union`, `sample`

**Wide** — each output partition depends on **many** input partitions. Data must be redistributed across the network — a **shuffle**. Wide transformations always create a new stage boundary.
- Examples: `groupBy`, `join`, `distinct`, `repartition`, `sortBy`

## The Full Execution Model

When you call an action, here is what happens end-to-end:

1. **Catalyst** parses your transformations into a logical plan
2. Catalyst resolves column names and types against the schema → **Analyzed logical plan**
3. Catalyst applies optimization rules (push predicates down, eliminate projections, etc.) → **Optimized logical plan**
4. Catalyst selects physical operators and costs them → **Physical plan**
5. **Tungsten** generates optimized JVM bytecode for the physical plan
6. The **DAG Scheduler** breaks the physical plan into a DAG of stages (cut at every shuffle)
7. The **Task Scheduler** assigns tasks to executor cores, preferring data-local placement
8. Executors run the tasks and return results / write output

### Adaptive Query Execution (AQE) — Spark 3.0+

AQE re-optimizes the query **at runtime** as real partition statistics become available — before Spark 3.0, all optimization happened upfront with estimated statistics.

| AQE Feature | What it does |
|---|---|
| **Dynamic coalescing** | Merges small post-shuffle partitions to avoid thousands of tiny tasks |
| **Dynamic join strategy** | Switches a sort-merge join to a broadcast join if one side turns out small |
| **Skew join handling** | Splits oversized partitions to prevent one slow task from blocking the whole stage |

AQE is enabled by default in Spark 3.2+. You can check with `spark.conf.get("spark.sql.adaptive.enabled")`.

## Hands-On: Seeing Lazy Evaluation in Action

In [ ]:
# Building a chain of lazy transformations — nothing executes yet
# Imagine this represents the card transaction processing pipeline
df = spark.range(0, 10_000_000)   # 10M rows representing transaction IDs

# These three lines are instant — Spark just records the plan
filtered  = df.filter(df.id % 2 == 0)                              # even-numbered txns
projected = filtered.selectExpr("id AS txn_id", "id * 2 AS ref_amount")
limited   = projected.limit(5)

print("No data has moved yet — just a logical plan.")
print(type(limited))


In [ ]:
# .show() is an action — this triggers execution
limited.show()   # now Spark runs the whole chain

In [ ]:
# Inspect the logical and physical plans
# Notice how Spark pushes the limit and filter as early as possible
limited.explain(mode="extended")

In [ ]:
# Wide transformation — groupBy causes a shuffle (stage boundary)
# Simulating: count card transactions per network bucket
from pyspark.sql.functions import count, col

wide = (
    spark.range(0, 1_000_000)
    .withColumn("network_idx", col("id") % 4)   # narrow — assign each txn to a network
    .filter(col("id") > 100)                     # narrow — discard test IDs
    .groupBy("network_idx")                      # wide   — shuffle boundary
    .agg(count("id").alias("txn_count"))
)
wide.show()


In [ ]:
# Check AQE status and inspect session config
print("AQE enabled :", spark.conf.get("spark.sql.adaptive.enabled"))
print("AQE coalesce:", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))
print("Shuffle parts:", spark.conf.get("spark.sql.shuffle.partitions"))

In [ ]:
# SparkSession.builder.getOrCreate() always returns the same session
spark2 = SparkSession.builder.appName("AnotherApp").getOrCreate()
print(spark is spark2)   # True — same session

## Summary

- `SparkSession` is the unified entry point since Spark 2.0, replacing `SQLContext`, `HiveContext`, and wrapping `SparkContext`.
- Build one with `SparkSession.builder.appName(...).master(...).getOrCreate()`; on Databricks, `spark` is always pre-created.
- Transformations are **lazy** — they build a logical plan without executing. Actions trigger execution.
- **Narrow transformations** (filter, select, map) keep data in the same partition — no shuffle, no stage boundary.
- **Wide transformations** (groupBy, join, distinct) redistribute data across the network — shuffle — and create a new stage.
- At execution time: Catalyst optimizes → Tungsten generates code → DAG Scheduler splits into stages → Task Scheduler assigns to executors.
- **Adaptive Query Execution** (Spark 3.0+) re-optimizes at runtime: coalesces small partitions, switches join strategies, handles skew.